TP4 RNN en NLP - Valentin MASSONNIERE

Sélectionnez un jeu de données textuel et prétraitez les données
Mon jeu de données : https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [1]:
# Load data

import pandas as pd

data = pd.read_csv("data/critics.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
# Put review in lowercase

data["review"] = data["review"].str.lower()
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [4]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt') 
nltk.download("punkt_tab")
nltk.download('wordnet')
nltk.download('stopwords') 
nltk.download('omw-1.4') 

# Tokenerization and revome ponctuation and HTML tags

data["tokens"] = data["review"].apply(word_tokenize)
data["clean_text"] = data["review"].apply(
    lambda x: re.sub(
    r'[^a-zA-Z\s]', 
    '', 
    re.sub(
        re.compile(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});'),
        '',
        x
        )
    )
)

swords = set(stopwords.words("english"))

data["tokens_no_stop"] = data["tokens"].apply(
    lambda tokens: [t for t in tokens if t.lower() not in swords]
)

# Lemmatisation

lemmatizer = WordNetLemmatizer()

data["lemmas"] = data["tokens_no_stop"].apply(
    lambda tokens: [lemmatizer.lemmatize(t) for t in tokens]
)

data["lemmas_str"] = data["lemmas"].apply(lambda lst: " ".join(lst))


# TF-IDF

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data["lemmas_str"])

data.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Valentin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Valentin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Valentin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Valentin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Valentin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,review,sentiment,tokens,clean_text,tokens_no_stop,lemmas,lemmas_str
0,one of the other reviewers has mentioned that ...,positive,"[one, of, the, other, reviewers, has, mentione...",one of the other reviewers has mentioned that ...,"[one, reviewers, mentioned, watching, 1, oz, e...","[one, reviewer, mentioned, watching, 1, oz, ep...",one reviewer mentioned watching 1 oz episode '...
1,a wonderful little production. <br /><br />the...,positive,"[a, wonderful, little, production, ., <, br, /...",a wonderful little production the filming tech...,"[wonderful, little, production, ., <, br, /, >...","[wonderful, little, production, ., <, br, /, >...",wonderful little production . < br / > < br / ...
2,i thought this was a wonderful way to spend ti...,positive,"[i, thought, this, was, a, wonderful, way, to,...",i thought this was a wonderful way to spend ti...,"[thought, wonderful, way, spend, time, hot, su...","[thought, wonderful, way, spend, time, hot, su...",thought wonderful way spend time hot summer we...
3,basically there's a family where a little boy ...,negative,"[basically, there, 's, a, family, where, a, li...",basically theres a family where a little boy j...,"[basically, 's, family, little, boy, (, jake, ...","[basically, 's, family, little, boy, (, jake, ...",basically 's family little boy ( jake ) think ...
4,"petter mattei's ""love in the time of money"" is...",positive,"[petter, mattei, 's, ``, love, in, the, time, ...",petter matteis love in the time of money is a ...,"[petter, mattei, 's, ``, love, time, money, ''...","[petter, mattei, 's, ``, love, time, money, ''...",petter mattei 's `` love time money '' visuall...


In [5]:
from sklearn.model_selection import train_test_split

# Split dataset into a train and test dataset with a good ratio random_state=42

X = vectorizer.fit_transform(data["lemmas_str"])
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    
    random_state=42,    
    stratify=y
)     

print(y_train.value_counts())
print(y_test.value_counts())

sentiment
positive    20000
negative    20000
Name: count, dtype: int64
sentiment
negative    5000
positive    5000
Name: count, dtype: int64


Concevez et implémentez un modèle RNN pour résoudre la tâche.